In [1]:
import pandas as pd
from urllib import request
import folium
from folium.plugins import TimestampedGeoJson
import json
from geojson import Point, Feature, FeatureCollection, dump
import pickle
from datetime import datetime, date
import os

# Get Data

Data was referred from Data is Plural via email subscription

Data is from [NYC Parks and Recreation - Street Tree Planting Locations](https://www.nycgovparks.org/trees/street-tree-planting/locations)



In [2]:
def get_most_recent_upload_date():
    r = request.urlopen('https://www.nycgovparks.org/tree-work-orders/street_tree_planting.csv')

    last_build_date = r.readlines()[5].decode('utf-8').split(',')[1].strip('\n')

    last_build_date = datetime.strptime(last_build_date.strip('"'), '%Y-%m-%d %H:%M:%S')

    return last_build_date

In [3]:
def download_new_data():

    df = pd.read_csv('https://www.nycgovparks.org/tree-work-orders/street_tree_planting.csv', skiprows=7)
    df['CompletedDate'] = pd.to_datetime(df['CompletedDate'])
    df['PlantingSeason'] = pd.to_datetime(df['PlantingSeason'])

    return df

In [4]:
def load_tree_data(save=True, return_data=True):

    today = datetime.now()
    file_names = os.listdir('../data/')
    today_file = f'../data/street_tree_planting_{today.strftime("%Y_%m_%d")}.pkl'

    if os.path.exists(today_file):
        df = pd.read_pickle(today_file)

    elif not os.path.exists(today_file):
        most_recent_web = get_most_recent_upload_date()
        if most_recent_web < today:
            most_recent_file = f'../data/street_tree_planting_{most_recent_web.strftime("%Y_%m_%d")}.pkl'
            if os.path.exists(most_recent_file):
                df = pd.read_pickle(most_recent_file)
            elif not os.path.exists(most_recent_file):
                df = download_new_data()
                if save:
                    df.to_pickle(f'../data/street_tree_planting_{most_recent_web.strftime("%Y_%m_%d")}.pkl')
        elif most_recent_web == today:
            df = get_most_recent_upload_date()
            if save:
                df.to_pickle(today_file)

    if return_data:
        return df


In [5]:
df = load_tree_data()
df

HTTPError: HTTP Error 403: Forbidden

In [6]:
nyc_map = folium.Map(location=[40.7128, -74.0060], zoom_start=12)
nyc_map

In [7]:
nyc_map = folium.Map(location=[40.7128, -74.0060], zoom_start=12)
records = df[df['WOStatus'] == 'Completed'][['lat', 'lng', 'CompletedDate']].to_records()

for record in records:
    folium.Marker(location=[record[1], record[2]],
                  icon=folium.Icon(color='green',
                                   icon_color='white',
                                   icon='tree',
                                   prefix='fa',
                                   ),
                  # popup=record[4],
                  tooltip=f"Completed Date: {record[3]}"
                  ).add_to(nyc_map)

nyc_map

In [7]:
def df_to_geojson(df, properties, lat='latitude', lon='longitude'):
    # create a new python dict to contain our geojson data, using geojson format
    geojson = {'type':'FeatureCollection', 'features':[]}

    # loop through each row in the dataframe and convert each row to geojson format
    for _, row in df.iterrows():
        # create a feature template to fill in
        feature = {'type':'Feature',
                   'properties':{},
                   'geometry':{'type':'Point',
                               'coordinates':[]}}

        # fill in the coordinates
        feature['geometry']['coordinates'] = [row[lon],row[lat]]

        # for each column, get the value and add it as a new feature property
        for prop in properties:
            feature['properties'][prop] = row[prop]

        # add this feature (aka, converted dataframe row) to the list of features inside our dict
        geojson['features'].append(feature)

    return geojson

In [13]:
len(df[df['WOStatus'] == 'Completed'])

6008

In [8]:
df_for_geojson = df[df['WOStatus'] == 'Completed'].rename({'lat': 'latitude', 'lng': 'longitude', 'CompletedDate': 'times'}, axis=1)
df_for_geojson['times'] = df_for_geojson['times'].astype(str)

geojson = df_to_geojson(df=df_for_geojson, properties=['times'], lat='latitude', lon='longitude')

In [9]:
with open('dataset.geojson', 'w') as f:
    dump(geojson, f)

In [19]:
with open('dataset.geojson') as j:
    data = json.load(j)

In [20]:
len(data['features'])

6008

In [21]:
data

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'properties': {'times': '2021-12-07'},
   'geometry': {'type': 'Point', 'coordinates': [-73.70950341, 40.74033958]}},
  {'type': 'Feature',
   'properties': {'times': '2021-12-07'},
   'geometry': {'type': 'Point', 'coordinates': [-73.7129842, 40.73542774]}},
  {'type': 'Feature',
   'properties': {'times': '2022-05-25'},
   'geometry': {'type': 'Point', 'coordinates': [-73.89458647, 40.72969599]}},
  {'type': 'Feature',
   'properties': {'times': '2022-03-07'},
   'geometry': {'type': 'Point', 'coordinates': [-73.92629176, 40.77010475]}},
  {'type': 'Feature',
   'properties': {'times': '2021-04-27'},
   'geometry': {'type': 'Point', 'coordinates': [-74.00817951, 40.67469629]}},
  {'type': 'Feature',
   'properties': {'times': '2021-04-27'},
   'geometry': {'type': 'Point', 'coordinates': [-74.01642919, 40.67630957]}},
  {'type': 'Feature',
   'properties': {'times': '2022-03-07'},
   'geometry': {'type': 'Point', 'coo

In [22]:
for i in range(len(data['features'])):
    data['features'][i]['properties']['times'] = [data['features'][i]['properties']['times'][:19]]

In [23]:
len(data['features'])

6008

In [24]:
data

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'properties': {'times': ['2021-12-07']},
   'geometry': {'type': 'Point', 'coordinates': [-73.70950341, 40.74033958]}},
  {'type': 'Feature',
   'properties': {'times': ['2021-12-07']},
   'geometry': {'type': 'Point', 'coordinates': [-73.7129842, 40.73542774]}},
  {'type': 'Feature',
   'properties': {'times': ['2022-05-25']},
   'geometry': {'type': 'Point', 'coordinates': [-73.89458647, 40.72969599]}},
  {'type': 'Feature',
   'properties': {'times': ['2022-03-07']},
   'geometry': {'type': 'Point', 'coordinates': [-73.92629176, 40.77010475]}},
  {'type': 'Feature',
   'properties': {'times': ['2021-04-27']},
   'geometry': {'type': 'Point', 'coordinates': [-74.00817951, 40.67469629]}},
  {'type': 'Feature',
   'properties': {'times': ['2021-04-27']},
   'geometry': {'type': 'Point', 'coordinates': [-74.01642919, 40.67630957]}},
  {'type': 'Feature',
   'properties': {'times': ['2022-03-07']},
   'geometry': {'type':

In [28]:
nyc_map = folium.Map(location=[40.7128, -74.0060], zoom_start=12)

TimestampedGeoJson(data, transition_time=50).add_to(nyc_map)

In [29]:
nyc_map

# Get Data Update

In [1]:
from requests import get
import pandas as pd
from datetime import datetime

In [2]:
def get_update_log():
    df_out = pd.read_csv('../data/last_update.csv')
    return df_out
# 
# update_log_df = get_update_log()
# update_log_df

In [3]:
# last_check_date = datetime.strptime(update_log_df['latest_check'][0], '%Y-%m-%d').date()
# if last_check_date < datetime.now().date():
#     print('yes')

In [4]:
def get_local_update_dates(in_df: pd.DataFrame) -> tuple:
    last_check_date = datetime.strptime(in_df['latest_check'][0], '%Y-%m-%d').date()
    last_web_date = datetime.strptime(in_df['latest_web_update'][0], '%Y-%m-%d').date()
    return last_check_date, last_web_date

In [12]:
def needs_web_update_check(last_check_date: datetime) -> bool:
    return last_check_date < datetime.now().date()
    # if last_check_date < datetime.now().date():
    #     return True
    # else:
    #     return False

In [6]:
def get_web_data():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }

    url = 'https://www.nycgovparks.org/tree-work-orders/street_tree_planting.csv'
    r = get(url, headers=headers)
    return r.text

In [7]:
def get_web_data_update_date(in_response_text: str) -> datetime:
    last_update = datetime.strptime(in_response_text.replace('"', '').split('\n')[5].split(',')[1], '%Y-%m-%d %H:%M:%S').date()
    return last_update

In [8]:
def transform_web_data(in_response_text: str):
    # Split the text into lines
    lines = r.text.replace('"', '').split('\n')
    
    # Extract the column names from line 7
    columns = lines[7].split(',')
    
    # Extract the data starting from line 8
    data = [line.split(',') for line in lines[8:] if line]
    
    # Create a DataFrame using the column names and data
    out_df = pd.DataFrame(data, columns=columns)
    
    return out_df

In [ ]:
def write_web_data(in_df: pd.DataFrame, last_web_date: datetime):
    in_df.to_pickle(f'data/street_tree_planting_{last_web_date.strftime("%Y_%m_%d")}.pkl')

In [9]:
def needs_web_data_update(local_web_update_date: datetime, web_update_date: datetime) -> bool:
    return local_web_update_date < web_update_date

In [10]:
def update_log(in_df: pd.DataFrame, last_check_date: datetime, last_web_date: datetime):
    in_df['latest_check'] = last_check_date
    in_df['latest_web_update'] = last_web_date
    in_df.to_csv('../data/last_update.csv', index=False)

In [ ]:
def update_data():
    log_df = get_update_log()
    latest_check_date, latest_web_update_date = get_local_update_dates(log_df)
    if needs_web_update_check(latest_check_date):
        latest_check_date = datetime.now().date().strftime('%Y-%m-%d')
        web_data_response = get_web_data()
        web_data_update_date = get_web_data_update_date(web_data_response)

        if needs_web_data_update(latest_web_update_date, web_data_update_date):
            new_web_data = transform_web_data(web_data_response)
            write_web_data(new_web_data, web_data_update_date)
            update_log(log_df, latest_check_date, web_data_update_date)

        else:
            update_log(log_df, latest_check_date, latest_web_update_date)

In [13]:
def main():
    update_data()
    

In [14]:
main()

NameError: name 'determine_data_update_required' is not defined

In [2]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

url = 'https://www.nycgovparks.org/tree-work-orders/street_tree_planting.csv'
r = get(url, headers=headers)
r

<Response [200]>

In [3]:
r.text.split('\n')[5].split(',')[1]

'"2024-08-05 00:50:18"'

In [4]:
last_update = datetime.strptime(r.text.replace('"', '').split('\n')[5].split(',')[1], '%Y-%m-%d %H:%M:%S').date()
if last_update < datetime.now().date():
    print('yes')

In [6]:
last_update.strftime('%Y-%m-%d')

'2024-08-03'

In [11]:
last_update

datetime.date(2024, 8, 3)

In [7]:
with open('../data/last_web_update.txt', 'w') as f:
    f.write(last_update.strftime('%Y-%m-%d'))

In [8]:
with open('../data/last_web_update.txt', 'r') as f:
    last_update_log_time = f.read()
    
last_update_log_time

'2024-08-03'

In [14]:
datetime.strptime(last_update_log_time, '%Y-%m-%d').date() == last_update

True

In [4]:
datetime.now().date().strftime('%Y-%m-%d')

'2024-08-05'

In [16]:
def get_most_recent_download_date():
    with open('../data/last_web_update.txt', 'r') as f:
        last_update_log_time = f.read()
    return datetime.strptime(last_update_log_time, '%Y-%m-%d').date()

last_local_data_update = get_most_recent_download_date()
last_local_data_update

datetime.date(2024, 8, 3)

In [ ]:
def download_new_data():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

    url = 'https://www.nycgovparks.org/tree-work-orders/street_tree_planting.csv'
    r = get(url, headers=headers)
    return r.text

In [ ]:
def main():
    last_local_data_update_date = get_most_recent_download_date()
    if last_local_data_update_date < datetime.now().date():
        new_content = download_new_data()
    
    last_web_data_update_date = get_most_recent_upload_date()
    if last_local_data_update_date < last_web_data_update_date:
        process_data(new_content)
    

In [ ]:
def check_stale_data(local_data_date, web_data_date):
    # local_data_date = get_most_recent_download_date()
    # if local_data_date < today:
    #     call download_new_data()
    
    if local_data_date < web_data_date:
        return True
    else:
        return False

In [ ]:
def check_last_web_update():
    last_update = datetime.strptime(r.text.replace('"', '').split('\n')[5].split(',')[1], '%Y-%m-%d %H:%M:%S').date()
    if last_update > last_update_log_time:
        return True
    else:
        return False

In [34]:
pd.DataFrame(r.text.replace('"', '').split('\n')[5].split(','), columns=['lastBuildDate'])#.iloc[1, :]

,lastBuildDate
0,lastBuildDate
1,2024-08-02 00:49:59


In [15]:
with open('street_tree_planting_2024.csv', 'wb') as f:
    f.write(r.content)

In [16]:
pd.read_csv('street_tree_planting_2024.csv', skiprows=7)

,lng,lat,Borough,ZipCode,BuildingNumber,StreetName,FiscalYear,PlantingSpaceID,CommunityBoard,PlantingSeason,CityCouncil,TreeID,WOId,WOStatus,CompletedDate
0,-73.903202,40.832201,Bronx,10456,585,EAST 169 STREET,0,8668134,203,06/30/2024,16,13927957,20147533,Completed,2023-05-24 00:00:00
1,-74.028143,40.629214,Brooklyn,11209,313,78 STREET,0,591706,310,06/30/2024,43,12164340,4887397,Completed,2021-12-06 00:00:00
2,-74.008408,40.630013,Brooklyn,11219,6502,10 AVENUE,0,7381440,310,06/30/2024,38,12164799,11537859,Completed,2021-12-06 00:00:00
3,-73.991873,40.627630,Brooklyn,11219,1515,57 STREET,0,4289309,312,06/30/2024,44,12167432,13987023,Completed,2021-12-06 00:00:00
4,-74.002787,40.743068,Manhattan,10011,350,WEST 18 STREET,0,717395,104,06/30/2024,3,583596,15947780,Not Completed,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10842,-73.935605,40.741249,Queens,11101,31-00,47 AVENUE,0,6439277,402,06/30/2024,26,0,19765016,Completed,2024-06-17 00:00:00
10843,-73.819840,40.586684,Queens,11693,320,BEACH 97 STREET,0,10006902,414,12/31/2024,32,0,18903413,Not Completed,NaN
10844,-73.947412,40.809936,Manhattan,10027,2133,ADAM C POWELL BOULEVARD,0,5813309,110,06/30/2024,9,0,7495306,Not Completed,NaN
10845,-73.735889,40.676076,Queens,11422,233-010,133 AVENUE,0,11474858,413,12/31/2024,31,0,22516799,Completed,2024-06-27 00:00:00


In [8]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

url = 'https://www.nycgovparks.org/tree-work-orders/street_tree_planting.csv'
r = get(url, headers=headers)

In [40]:
# Split the text into lines
lines = r.text.replace('"', '').split('\n')

# Extract the column names from line 7
columns = lines[7].split(',')

# Extract the data starting from line 8
data = [line.split(',') for line in lines[8:] if line]

# Create a DataFrame using the column names and data
# df = 
pd.DataFrame(data, columns=columns)

# # Display the DataFrame
# df.head()
# 
# 
# pd.DataFrame([line.split(',') for line in r.text.replace('"', '').split('\n')[7:] if line])

,lng,lat,Borough,ZipCode,BuildingNumber,StreetName,FiscalYear,PlantingSpaceID,CommunityBoard,PlantingSeason,CityCouncil,TreeID,WOId,WOStatus,CompletedDate
0,-73.90320165,40.83220053,Bronx,10456,585,EAST 169 STREET,0,8668134,203,06/30/2024,16,13927957,20147533,Completed,2023-05-24 00:00:00
1,-74.02814289,40.62921415,Brooklyn,11209,313,78 STREET,0,591706,310,06/30/2024,43,12164340,4887397,Completed,2021-12-06 00:00:00
2,-74.00840803,40.63001274,Brooklyn,11219,6502,10 AVENUE,0,7381440,310,06/30/2024,38,12164799,11537859,Completed,2021-12-06 00:00:00
3,-73.99187301,40.62763026,Brooklyn,11219,1515,57 STREET,0,4289309,312,06/30/2024,44,12167432,13987023,Completed,2021-12-06 00:00:00
4,-74.00278706,40.74306825,Manhattan,10011,350,WEST 18 STREET,0,717395,104,06/30/2024,3,583596,15947780,Not Completed,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10760,-73.99941761,40.71195109,Manhattan,10038,27,ST JAMES PLACE,0,428471,103,06/30/2024,1,0,780220,Not Completed,
10761,-73.81984044,40.58668423,Queens,11693,320,BEACH 97 STREET,0,10006902,414,12/31/2024,32,0,18903413,Not Completed,
10762,-73.94741160,40.80993558,Manhattan,10027,2133,ADAM C POWELL BOULEVARD,0,5813309,110,06/30/2024,9,0,7495306,Not Completed,
10763,-73.73588904,40.67607602,Queens,11422,233-010,133 AVENUE,0,11474858,413,12/31/2024,31,0,22516799,Completed,2024-06-27 00:00:00


In [1]:
import pandas as pd

In [2]:
df = pd.read_pickle('../data/street_tree_planting_2022_08_03.pkl')
df

,lng,lat,Borough,ZipCode,BuildingNumber,StreetName,FiscalYear,PlantingSpaceID,CommunityBoard,PlantingSeason,CityCouncil,TreeID,WOId,WOStatus,CompletedDate
0,-73.709503,40.740340,Queens,11004,82-030,260 STREET,0,148027,413,2022-05-31,23,114029,15525248,Completed,2021-12-07
1,-73.954008,40.814634,Manhattan,10027,464,WEST 129 STREET,0,156427,109,2022-05-31,7,0,16644663,Not Completed,NaT
2,-73.712984,40.735428,Queens,11001,254-04,84 ROAD,0,177638,413,2022-05-31,23,142438,8573730,Completed,2021-12-07
3,-73.952187,40.799678,Manhattan,10026,37,MALCOLM X BOULEVARD,0,178031,110,2022-05-31,9,143233,9138879,Not Completed,NaT
4,-73.876284,40.828165,Bronx,10472,1155,MANOR AVENUE,0,191641,209,2022-05-31,18,0,14343624,Not Completed,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8180,-73.785172,40.727778,Queens,11366,183-02,UNION TURNPIKE,0,4309386,408,2022-05-31,24,13133256,12931606,Completed,2022-05-24
8181,-73.810892,40.777863,Queens,11357,151-41,WILLETS POINT BOULEVARD,0,4995347,407,2022-05-31,19,13133260,14373476,Completed,2022-05-09
8182,-73.708925,40.743219,Queens,11004,81-04,262 STREET,0,4302751,413,2022-05-31,23,13133270,16743002,Completed,2022-04-21
8183,-73.870720,40.896687,Bronx,10470,143,EAST 233 STREET,0,5803374,212,2022-05-31,11,13133286,15866418,Completed,2022-06-28


In [3]:
df2 = pd.read_pickle('../data/street_tree_planting_2024_08_12.pkl')
df2

,lng,lat,Borough,ZipCode,BuildingNumber,StreetName,FiscalYear,PlantingSpaceID,CommunityBoard,PlantingSeason,CityCouncil,TreeID,WOId,WOStatus,CompletedDate
0,-73.90320165,40.83220053,Bronx,10456,585,EAST 169 STREET,0,8668134,203,2024-06-30,16,13927957,20147533,Completed,2023-05-24
1,-74.02814289,40.62921415,Brooklyn,11209,313,78 STREET,0,591706,310,2024-06-30,43,12164340,4887397,Completed,2021-12-06
2,-74.00840803,40.63001274,Brooklyn,11219,6502,10 AVENUE,0,7381440,310,2024-06-30,38,12164799,11537859,Completed,2021-12-06
3,-73.99187301,40.62763026,Brooklyn,11219,1515,57 STREET,0,4289309,312,2024-06-30,44,12167432,13987023,Completed,2021-12-06
4,-74.00278706,40.74306825,Manhattan,10011,350,WEST 18 STREET,0,717395,104,2024-06-30,3,583596,15947780,Not Completed,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10761,-73.99941761,40.71195109,Manhattan,10038,27,ST JAMES PLACE,0,428471,103,2024-06-30,1,0,780220,Not Completed,NaT
10762,-73.81984044,40.58668423,Queens,11693,320,BEACH 97 STREET,0,10006902,414,2024-12-31,32,0,18903413,Not Completed,NaT
10763,-73.94741160,40.80993558,Manhattan,10027,2133,ADAM C POWELL BOULEVARD,0,5813309,110,2024-06-30,9,0,7495306,Not Completed,NaT
10764,-73.73588904,40.67607602,Queens,11422,233-010,133 AVENUE,0,11474858,413,2024-12-31,31,0,22516799,Completed,2024-06-27
